# Text Mining Experiments (Indonesian)
1. Hate Speech Classification (Sentence-BERT + MLP)
2. Persona Analysis of Tweets (LSA + K-Means, LDA)
3. Text Summarization (Extractive, TextRank)

Outputs: images/CSVs in `../result`, best models in `../result/model`.

## Shared Setup

In [ ]:
# !pip install -U sentence-transformers optuna rouge wordcloud networkx keras joblib
import re
import warnings
from pathlib import Path

import nltk

warnings.filterwarnings("ignore")
for pkg in ["stopwords", "punkt", "punkt_tab"]:
    nltk.download(pkg, quiet=True)

DATA_DIR = Path("../data")
RESULT_DIR = Path("../result")
MODEL_DIR = RESULT_DIR / "model"
CACHE_DIR = RESULT_DIR / "cache"
for d in (MODEL_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
SEED = 42

SLANG = {
    "ad": "ada",
    "ap": "apa",
    "adlh": "adalah",
    "dg": "dengan",
    "dgn": "dengan",
    "bgt": "banget",
    "dri": "dari",
    "emg": "emang",
    "gt": "gitu",
    "jd": "jadi",
    "jdi": "jadi",
    "jg": "juga",
    "jgn": "jangan",
    "klo": "kalau",
    "sdh": "sudah",
    "tau": "tahu",
    "tdk": "tidak",
    "tpi": "tapi",
    "utk": "untuk",
    "yg": "yang",
}
NOISE = re.compile(r"http\S+|www\.\S+|@\w+|#\w+|&\w+;|\b(?:rt|url|user)\b")


def clean_text(texts):
    """Lowercase, strip URLs/mentions/hashtags/HTML entities/non-letters, normalize slang."""
    out = []
    for t in texts:
        t = re.sub(r"[^a-z\s]", " ", NOISE.sub(" ", str(t).lower()))
        out.append(" ".join(SLANG.get(w, w) for w in t.split()))
    return out

# Case 1: Hate Speech Classification

In [ ]:
import json

import keras
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from keras import callbacks, layers
from sentence_transformers import SentenceTransformer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
DATA_PATH_1 = DATA_DIR / "data_1A.csv"
EMBED_MODEL = "firqaaa/indo-sentence-bert-base"
EMB_CACHE = CACHE_DIR / "case1_embeddings.npy"
MODEL_PATH_1 = MODEL_DIR / "case1_hate_speech.keras"
THRESH_PATH_1 = MODEL_DIR / "case1_thresholds.json"
LABELS = [
    "HS",
    "Abusive",
    "HS_Individual",
    "HS_Group",
    "HS_Religion",
    "HS_Race",
    "HS_Physical",
    "HS_Gender",
    "HS_Other",
]
N_TRIALS, EPOCHS = 15, 30

In [ ]:
df_1 = pd.read_csv(DATA_PATH_1, index_col=0)
df_1["Clean_Tweet"] = clean_text(df_1["Tweet"])
print(df_1.shape)
print(df_1[LABELS].sum().sort_values(ascending=False))

In [ ]:
# Embeddings are cached, the slowest step on CPU
embedder = SentenceTransformer(EMBED_MODEL)
if EMB_CACHE.exists():
    X = np.load(EMB_CACHE)
else:
    X = embedder.encode(
        df_1["Clean_Tweet"].tolist(),
        batch_size=64,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    np.save(EMB_CACHE, X)
y = df_1[LABELS].to_numpy()

# 80/10/10 split stratified on HS
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y[:, 0]
)
X_va, X_te, y_va, y_te = train_test_split(
    X_te, y_te, test_size=0.5, random_state=SEED, stratify=y_te[:, 0]
)

In [ ]:
def suggest(t):
    p = {
        "n_layers": t.suggest_int("n_layers", 1, 2),
        "act": t.suggest_categorical("act", ["relu", "gelu"]),
        "lr": t.suggest_float("lr", 1e-4, 3e-3, log=True),
        "batch": t.suggest_categorical("batch", [32, 64, 128]),
    }
    for i in range(p["n_layers"]):
        p[f"units_{i}"] = t.suggest_int(f"units_{i}", 64, 512, log=True)
        p[f"drop_{i}"] = t.suggest_float(f"drop_{i}", 0.1, 0.5)
    return p


def train(p):
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEED)
    m = keras.Sequential([keras.Input((X.shape[1],))])
    for i in range(p["n_layers"]):
        m.add(layers.Dense(p[f"units_{i}"], activation=p["act"]))
        m.add(layers.Dropout(p[f"drop_{i}"]))
    m.add(layers.Dense(len(LABELS), activation="sigmoid"))
    m.compile(keras.optimizers.Adam(p["lr"]), "binary_crossentropy")
    cb = [
        callbacks.EarlyStopping(patience=3, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(factor=0.3, patience=1),
    ]
    h = m.fit(
        X_tr,
        y_tr,
        validation_data=(X_va, y_va),
        epochs=EPOCHS,
        batch_size=p["batch"],
        callbacks=cb,
        verbose=0,
    )
    return m, h


def objective(t):  # optimize the target metric directly
    m, _ = train(suggest(t))
    return f1_score(
        y_va, m.predict(X_va, verbose=0) >= 0.5, average="weighted", zero_division=0
    )


study = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(objective, n_trials=N_TRIALS)
print(f"Best val F1: {study.best_value:.3f}\n{study.best_params}")
optuna.visualization.matplotlib.plot_optimization_history(study)

plt.savefig(RESULT_DIR / "case1_optuna_history.png", bbox_inches="tight")
plt.show()

In [ ]:
model_1, hist = train(suggest(optuna.trial.FixedTrial(study.best_params)))

# Per-label decision threshold tuned on validation set
p_va = model_1.predict(X_va, verbose=0)
grid = np.arange(0.1, 0.9, 0.05)
thresholds = [
    float(
        grid[
            np.argmax(
                [f1_score(y_va[:, j], p_va[:, j] >= g, zero_division=0) for g in grid]
            )
        ]
    )
    for j in range(len(LABELS))
]

pred = (model_1.predict(X_te, verbose=0) >= thresholds).astype(int)
for name, fn in [
    ("Accuracy", accuracy_score),
    ("Precision", precision_score),
    ("Recall", recall_score),
    ("F1", f1_score),
]:
    kw = {} if fn is accuracy_score else {"average": "weighted", "zero_division": 0}
    print(f"{name}: {fn(y_te, pred, **kw):.3f}")
print(classification_report(y_te, pred, target_names=LABELS, zero_division=0))

plt.figure(figsize=(5, 4))
plt.plot(hist.history["loss"], label="train")
plt.plot(hist.history["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.savefig(RESULT_DIR / "case1_loss.png", bbox_inches="tight")
plt.show()

model_1.save(MODEL_PATH_1)
THRESH_PATH_1.write_text(json.dumps(thresholds))

### Inference

In [ ]:
clf = keras.models.load_model(MODEL_PATH_1)
th = np.array(json.loads(THRESH_PATH_1.read_text()))


def predict_hate(texts):
    emb = embedder.encode(clean_text(texts), normalize_embeddings=True)
    return pd.DataFrame(
        (clf.predict(emb, verbose=0) >= th).astype(int), columns=LABELS, index=texts
    )


predict_hate(
    ["Dasar cebong dungu, otak udang!", "Selamat pagi, semoga harimu menyenangkan"]
)

# Case 2: Persona Analysis of Tweets

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from wordcloud import WordCloud

In [ ]:
DATA_PATH_2 = DATA_DIR / "data_2A.csv"
MODEL_PATH_2 = MODEL_DIR / "case2_kmeans.joblib"
STOPWORDS_ID = set(stopwords.words("indonesian")) | {
    "aja",
    "ga",
    "gak",
    "nya",
    "sih",
    "deh",
    "kok",
    "ya",
    "yg",
    "kalo",
    "gitu",
    "banget",
}

K_RANGE = range(2, 11)
SVD_DIM, TOP_WORDS, LDA_PASSES = 100, 10, 10

In [ ]:
df_2 = pd.read_csv(DATA_PATH_2, index_col=0)[["Tweet"]]
df_2["Clean_Tweet"] = clean_text(df_2["Tweet"])
df_2 = df_2[df_2["Clean_Tweet"].str.len() > 0].reset_index(drop=True)

# TF-IDF -> LSA (SVD + L2 norm): denser space, faster and better-separated clusters than raw sparse TF-IDF
tfidf = TfidfVectorizer(
    stop_words=list(STOPWORDS_ID),
    min_df=3,
    max_df=0.5,
    ngram_range=(1, 2),
    sublinear_tf=True,
)
lsa = make_pipeline(TruncatedSVD(SVD_DIM, random_state=SEED), Normalizer())
Z = lsa.fit_transform(tfidf.fit_transform(df_2["Clean_Tweet"]))

inertia, sil = [], []
for k in K_RANGE:
    km = KMeans(k, n_init=10, random_state=SEED).fit(Z)
    inertia.append(km.inertia_)
    sil.append(
        silhouette_score(
            Z, km.labels_, sample_size=min(5000, len(Z)), random_state=SEED
        )
    )
best_k = K_RANGE[int(np.argmax(sil))]
print(f"Best k = {best_k} (silhouette {max(sil):.3f})")

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(K_RANGE, inertia, "bx-")
ax[0].set(title="Elbow", xlabel="k", ylabel="Inertia")
ax[1].plot(K_RANGE, sil, "ro-")
ax[1].set(title="Silhouette", xlabel="k", ylabel="Score")
plt.tight_layout()
plt.savefig(RESULT_DIR / "case2_k_selection.png", bbox_inches="tight")
plt.show()

In [ ]:
model_2 = make_pipeline(
    tfidf, *[s for _, s in lsa.steps], KMeans(best_k, n_init=10, random_state=SEED)
)
df_2["cluster"] = model_2.fit_predict(df_2["Clean_Tweet"])
df_2.to_csv(RESULT_DIR / "case2_clusters.csv", index=False)
print(df_2["cluster"].value_counts().sort_index())

# Word clouds from centroids projected back to term space
km, svd, terms = model_2[-1], model_2[1], model_2[0].get_feature_names_out()
centroids = svd.inverse_transform(km.cluster_centers_)
cols = 4
rows = -(-best_k // cols)
plt.figure(figsize=(4 * cols, 3 * rows))
for c, w in enumerate(centroids):
    top = w.argsort()[::-1][:TOP_WORDS]
    plt.subplot(rows, cols, c + 1)
    plt.imshow(
        WordCloud(background_color="white").generate_from_frequencies(
            {terms[i]: max(w[i], 1e-6) for i in top}
        )
    )
    plt.title(f"Cluster {c}")
    plt.axis("off")
plt.tight_layout()
plt.savefig(RESULT_DIR / "case2_wordclouds.png", bbox_inches="tight")
plt.show()

joblib.dump(model_2, MODEL_PATH_2)

In [ ]:
# LDA topics as a complementary view
cv = CountVectorizer(stop_words=list(STOPWORDS_ID), min_df=5, max_df=0.5)
lda = LatentDirichletAllocation(
    best_k, max_iter=LDA_PASSES, learning_method="online", random_state=SEED, n_jobs=-1
)
lda.fit(cv.fit_transform(df_2["Clean_Tweet"]))

vocab = cv.get_feature_names_out()
for tid, w in enumerate(lda.components_):
    print(f"Topic {tid}: {list(vocab[w.argsort()[::-1][:TOP_WORDS]])}")

### Inference

In [ ]:
km_pipe = joblib.load(MODEL_PATH_2)
samples = [
    "@bankbsi_id kok mobile banking gabisa dipake dari kemarin",
    "Transfer saldo sukses, terima kasih",
]

pd.DataFrame({"Tweet": samples, "cluster": km_pipe.predict(clean_text(samples))})

# Case 3: Text Summarization (Extractive, TextRank)

In [ ]:
import json
import re

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from nltk import sent_tokenize
from nltk.corpus import stopwords
from rouge import Rouge
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
DATA_PATH_3 = DATA_DIR / "data_3A.csv"
CONFIG_PATH_3 = MODEL_DIR / "case3_textrank.json"
STOPWORDS_SUM = stopwords.words("indonesian") + [
    "baiknya",
    "berkali",
    "kali",
    "kurangnya",
    "mata",
    "olah",
    "sekurang",
    "setidak",
    "tama",
    "tidaknya",
]

TOP_NS = range(1, 6)

In [ ]:
def clean_article(t):
    t = re.sub(r"^.*?TEMPO\.CO,\s*\w+\s*-\s*", "", t).split("Pilihan Editor:")[0]
    return re.sub(
        r'([.!?]["”]?)(?=["“A-Z])', r"\1 ", t
    ).strip()  # fix missing space between sentences


def summarize(text, top_n):
    sents = sent_tokenize(text)
    if len(sents) <= top_n:
        return text
    sim = cosine_similarity(
        TfidfVectorizer(stop_words=STOPWORDS_SUM).fit_transform(sents)
    )
    np.fill_diagonal(sim, 0)
    scores = nx.pagerank(nx.from_numpy_array(sim))
    top = sorted(
        sorted(scores, key=scores.get, reverse=True)[:top_n]
    )  # keep original order
    return " ".join(sents[i] for i in top)


df_3 = pd.read_csv(DATA_PATH_3, index_col=0).reset_index(drop=True)
df_3["content"] = df_3["content"].apply(clean_article)

rouge, rows = Rouge(), []
for n in TOP_NS:
    df_3[f"top_{n}"] = df_3["content"].apply(summarize, top_n=n)
    s = rouge.get_scores(df_3[f"top_{n}"].tolist(), df_3["summary"].tolist(), avg=True)
    rows.append({"top_n": n, **{k: v["f"] for k, v in s.items()}})
scores = pd.DataFrame(rows).set_index("top_n")
best_n = int(scores["rouge-l"].idxmax())
print(scores.round(3), f"\nBest top_n = {best_n}")

scores.plot.bar(figsize=(6, 4), title="ROUGE F1 by number of sentences")
plt.ylabel("F1")
plt.savefig(RESULT_DIR / "case3_rouge.png", bbox_inches="tight")
plt.show()
df_3.to_csv(RESULT_DIR / "case3_summaries.csv", index=False)
CONFIG_PATH_3.write_text(
    json.dumps({"top_n": best_n, "rouge": scores.loc[best_n].to_dict()})
)

### Inference

In [ ]:
cfg = json.loads(CONFIG_PATH_3.read_text())
article = (
    "TEMPO.CO, Jakarta - Pemerintah resmi menaikkan harga BBM bersubsidi mulai hari ini. "
    "Menteri ESDM mengatakan kenaikan ini diperlukan untuk menjaga APBN. "
    "Sejumlah warga mengeluhkan kenaikan tersebut karena berdampak pada harga kebutuhan pokok. "
    "Pemerintah berjanji menyalurkan bantuan langsung tunai kepada masyarakat miskin. "
    "Pengamat menilai bantuan tersebut harus tepat sasaran agar efektif."
)

print(summarize(clean_article(article), cfg["top_n"]))